# SleepPy Extraction Review

This notebook reviews first-pass sleep metric extraction from Oura Ring 4 finger, Oura Ring 3 toe, Samsung Watch / SleepWatch, Muse, and OSCAR/SleepScope sample screenshots or PDFs.

**Disclaimer:** This is exploratory wellness data analysis only. It is not medical diagnosis, treatment advice, or a replacement for clinician review.

## Workflow

1. Place 1-2 representative files per device under `data/raw/samples/<device>/`.
2. Run the extraction cell to create `data/processed/nightly_summary.csv`, `data/processed/device_observations_long.csv`, and `outputs/extraction_report.md`.
3. Review extracted values, missingness, confidence labels, and trend plots.
4. Correct low-confidence or missing values manually before treating them as analysis inputs.

In [19]:
from pathlib import Path
import importlib

import matplotlib.pyplot as plt
import pandas as pd

# Reload local modules so an already-running notebook kernel picks up edits from disk.
import sleeppy.schema as sleep_schema
import sleeppy.quality as sleep_quality
import sleeppy.extract as sleep_extract
import sleeppy.extract.common as extract_common
import sleeppy.extract.pipeline as extract_pipeline

importlib.reload(sleep_schema)
importlib.reload(sleep_quality)
importlib.reload(extract_common)
importlib.reload(extract_pipeline)
importlib.reload(sleep_extract)

check_ocr_environment = extract_common.check_ocr_environment
run_sample_extraction = extract_pipeline.run_sample_extraction
confidence_by_device = sleep_quality.confidence_by_device
describe_extraction_outputs = sleep_quality.describe_extraction_outputs
missingness_by_device = sleep_quality.missingness_by_device
observations_to_nightly_summary = sleep_quality.observations_to_nightly_summary
CANONICAL_METRIC_UNITS = sleep_schema.CANONICAL_METRIC_UNITS
PLOT_METRICS = sleep_schema.PLOT_METRICS
ensure_observations_frame = sleep_schema.ensure_observations_frame
normalize_summary_columns = sleep_schema.normalize_summary_columns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## OCR Diagnostics

The notebook must have dependencies installed in the active kernel shown below. If `image_ocr_ready` is `False`, run `import sys; !"{sys.executable}" -m pip install -r requirements.txt`, then restart the kernel.

In [20]:
ocr_status = check_ocr_environment()
display(pd.DataFrame([ocr_status]))

if not ocr_status["image_ocr_ready"]:
    print("Image OCR is not ready in this active notebook kernel. Screenshots will produce zero extracted values until this is fixed.")
    print(ocr_status["notes"])
else:
    print("Image OCR is ready.")

,python_executable,pillow_installed,pytesseract_installed,pymupdf_installed,tesseract_cmd,tesseract_version,image_ocr_ready,pdf_text_ready,notes
0,C:\Users\morri\miniconda3\python.exe,True,True,True,C:\Program Files\Tesseract-OCR\tesseract.exe,5.4.0.20240606,True,True,OCR/PDF dependencies look ready.


Image OCR is ready.


In [21]:
RUN_EXTRACTION = True

if RUN_EXTRACTION:
    nightly_summary, observations, report_path = run_sample_extraction(
        raw_samples_dir=PROJECT_ROOT / "data" / "raw" / "samples",
        processed_dir=PROCESSED_DIR,
        outputs_dir=OUTPUTS_DIR,
        include_legacy_raw=True,
    )
else:
    nightly_summary = pd.read_csv(PROCESSED_DIR / "nightly_summary.csv")
    observations = pd.read_csv(PROCESSED_DIR / "device_observations_long.csv")
    report_path = OUTPUTS_DIR / "extraction_report.md"

observations = ensure_observations_frame(observations)
if observations.empty:
    nightly_summary = normalize_summary_columns(nightly_summary)
else:
    nightly_summary = observations_to_nightly_summary(observations)

print(f"Night/device rows: {len(nightly_summary)}")
print(f"Extracted values: {len(observations)}")
print(f"Report: {report_path}")
print("Available observation metrics:", sorted(observations["metric"].dropna().unique().tolist()))
print("Nightly summary columns:", list(nightly_summary.columns))

extraction_diagnostics = describe_extraction_outputs(nightly_summary, observations)

Night/device rows: 2
Extracted values: 18
Report: C:\Users\morri\PycharmProjects\SleepPy\outputs\extraction_report.md
Available observation metrics: ['avg_hr_bpm', 'avg_hrv_ms', 'avg_spo2_pct', 'awake_minutes', 'breathing_label', 'deep_minutes', 'light_minutes', 'min_hr_bpm', 'rem_minutes', 'respiratory_rate_bpm', 'sleep_efficiency_pct', 'time_in_bed_minutes', 'total_sleep_minutes']
Nightly summary columns: ['night_date', 'device', 'total_sleep_minutes', 'time_in_bed_minutes', 'sleep_efficiency_pct', 'sleep_score', 'avg_hr_bpm', 'min_hr_bpm', 'avg_hrv_ms', 'avg_spo2_pct', 'min_spo2_pct', 'respiratory_rate_bpm', 'temperature_deviation_c', 'readiness_score', 'cpap_ahi', 'cpap_usage_hours', 'cpap_leak_rate', 'cpap_pressure', 'awake_minutes', 'rem_minutes', 'light_minutes', 'deep_minutes', 'max_hrv_ms', 'hrv_balance_score', 'breathing_label', 'cpap_cai', 'cpap_oai', 'source_files', 'extraction_methods', 'min_confidence', 'notes']
Night/device rows: 2
Observation rows: 18
Devices detected: 

## Extracted Tables

In [22]:
display(nightly_summary)
display(observations.head(50))

,night_date,device,total_sleep_minutes,time_in_bed_minutes,sleep_efficiency_pct,sleep_score,avg_hr_bpm,min_hr_bpm,avg_hrv_ms,avg_spo2_pct,min_spo2_pct,respiratory_rate_bpm,temperature_deviation_c,readiness_score,cpap_ahi,cpap_usage_hours,cpap_leak_rate,cpap_pressure,awake_minutes,rem_minutes,light_minutes,deep_minutes,max_hrv_ms,hrv_balance_score,breathing_label,cpap_cai,cpap_oai,source_files,extraction_methods,min_confidence,notes
0,2026-06-08,Samsung Watch / SleepWatch,NaN,NaN,NaN,<NA>,57,NaN,NaN,NaN,<NA>,11.5,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,<NA>,<NA>,data\raw\samples\samsung_watch\Screenshot_2026...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
1,undated,Oura Ring 4 finger,381,381,89,<NA>,NaN,54,20,6,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,48,73,265,43,<NA>,<NA>,disturbances were,<NA>,<NA>,data\raw\samples\oura4\0EB30769-382F-4A4B-A4AA...,ocr,low,OCR extracted with pytesseract. Using Tesserac...


,night_date,device,metric,value,unit,source_file,extraction_method,confidence,notes
0,<NA>,Oura Ring 4 finger,awake_minutes,48,minutes,data\raw\samples\oura4\0EB30769-382F-4A4B-A4AA...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
1,<NA>,Oura Ring 4 finger,rem_minutes,73,minutes,data\raw\samples\oura4\0EB30769-382F-4A4B-A4AA...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
2,<NA>,Oura Ring 4 finger,light_minutes,265,minutes,data\raw\samples\oura4\0EB30769-382F-4A4B-A4AA...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
3,<NA>,Oura Ring 4 finger,deep_minutes,43,minutes,data\raw\samples\oura4\0EB30769-382F-4A4B-A4AA...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
4,<NA>,Oura Ring 4 finger,total_sleep_minutes,381,minutes,data\raw\samples\oura4\37A28C13-74A7-4059-B80D...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
5,<NA>,Oura Ring 4 finger,sleep_efficiency_pct,89,pct,data\raw\samples\oura4\37A28C13-74A7-4059-B80D...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
6,<NA>,Oura Ring 4 finger,min_hr_bpm,54,bpm,data\raw\samples\oura4\6B3E3B9B-1F0D-4EE7-B182...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
7,<NA>,Oura Ring 4 finger,avg_hrv_ms,20,ms,data\raw\samples\oura4\6B3E3B9B-1F0D-4EE7-B182...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
8,<NA>,Oura Ring 4 finger,total_sleep_minutes,381,minutes,data\raw\samples\oura4\933CACBF-2408-46D2-923F...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...
9,<NA>,Oura Ring 4 finger,sleep_efficiency_pct,89,pct,data\raw\samples\oura4\933CACBF-2408-46D2-923F...,ocr,medium,OCR extracted with pytesseract. Using Tesserac...


## Missingness and Confidence

In [23]:
missingness = missingness_by_device(nightly_summary)
confidence = confidence_by_device(observations)

if missingness.empty:
    print("No nightly summary rows available yet.")
else:
    display(missingness.pivot_table(index="metric", columns="device", values="missing_pct"))

if confidence.empty:
    print("No extracted observations available yet.")
else:
    display(confidence.pivot_table(index="device", columns="confidence", values="count", fill_value=0))

device,Oura Ring 4 finger,Samsung Watch / SleepWatch
metric,,
avg_hr_bpm,100.0,0.0
avg_hrv_ms,0.0,100.0
avg_spo2_pct,0.0,100.0
awake_minutes,0.0,100.0
breathing_label,0.0,100.0
cpap_ahi,100.0,100.0
cpap_cai,100.0,100.0
cpap_leak_rate,100.0,100.0
cpap_oai,100.0,100.0


confidence,low,medium
device,,
Oura Ring 4 finger,1.0,15.0
Samsung Watch / SleepWatch,0.0,2.0


## Trends

In [24]:
PLOT_SPECS = {
    "total_sleep_minutes": ("Minutes", "Sleep duration"),
    "avg_hrv_ms": ("ms", "Average HRV"),
    "avg_spo2_pct": ("%", "Average SpO2"),
    "cpap_ahi": ("Events/hour", "CPAP AHI"),
}


def available_numeric_metrics(summary_df):
    summary_df = normalize_summary_columns(summary_df)
    metrics = []
    for metric in CANONICAL_METRIC_UNITS:
        if metric in summary_df and pd.to_numeric(summary_df[metric], errors="coerce").notna().any():
            metrics.append(metric)
    return metrics


def plot_metric(summary_df, metric, ylabel, title):
    summary_df = normalize_summary_columns(summary_df)
    if summary_df.empty or metric not in summary_df:
        print(f"No {metric} column available. Available numeric metrics: {available_numeric_metrics(summary_df)}")
        return None

    df = summary_df.copy()
    df["parsed_night_date"] = pd.to_datetime(df["night_date"], errors="coerce")
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    has_metric_values = df[metric].notna().any()
    plottable = df.dropna(subset=["parsed_night_date", metric])
    if plottable.empty:
        if has_metric_values:
            print(f"{metric} values exist, but none have a parseable night_date for trend plotting.")
        else:
            print(f"No plottable {metric} values available. Available numeric metrics: {available_numeric_metrics(summary_df)}")
        return None

    df = plottable
    if df.empty:
        print(f"No plottable {metric} values available. Available numeric metrics: {available_numeric_metrics(summary_df)}")
        return None

    fig, ax = plt.subplots(figsize=(10, 4))
    for device, group in df.groupby("device", sort=True):
        group = group.sort_values("parsed_night_date")
        ax.plot(group["parsed_night_date"], group[metric], marker="o", label=device)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Date")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best")
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig, ax

In [25]:
for metric in PLOT_METRICS:
    ylabel, title = PLOT_SPECS.get(metric, (CANONICAL_METRIC_UNITS.get(metric, "Value"), metric))
    plot_metric(nightly_summary, metric, ylabel, title)
plt.show()

total_sleep_minutes values exist, but none have a parseable night_date for trend plotting.
avg_hrv_ms values exist, but none have a parseable night_date for trend plotting.
avg_spo2_pct values exist, but none have a parseable night_date for trend plotting.
No plottable cpap_ahi values available. Available numeric metrics: ['total_sleep_minutes', 'time_in_bed_minutes', 'sleep_efficiency_pct', 'avg_hr_bpm', 'min_hr_bpm', 'avg_hrv_ms', 'avg_spo2_pct', 'respiratory_rate_bpm', 'awake_minutes', 'rem_minutes', 'light_minutes', 'deep_minutes']


## Extraction Report

In [26]:
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("No extraction report found yet.")

# SleepPy Extraction Report

This report summarizes automated first-pass extraction from screenshots and PDFs.

This is exploratory wellness data analysis only. It is not medical diagnosis, treatment advice, or a replacement for clinician review.

- Night/device rows: 2
- Extracted values: 18

## Values By Device

- Oura Ring 4 finger: 16
- Samsung Watch / SleepWatch: 2

## Canonical Metrics Available

- total_sleep_minutes
- time_in_bed_minutes
- sleep_efficiency_pct
- avg_hr_bpm
- min_hr_bpm
- avg_hrv_ms
- avg_spo2_pct
- respiratory_rate_bpm
- awake_minutes
- rem_minutes
- light_minutes
- deep_minutes
- breathing_label

## Confidence By Device

- Oura Ring 4 finger: low = 1
- Oura Ring 4 finger: medium = 15
- Samsung Watch / SleepWatch: medium = 2

## File Notes

- OCR environment: python=C:\Users\morri\miniconda3\python.exe; pillow=True; pytesseract=True; tesseract_cmd=C:\Program Files\Tesseract-OCR\tesseract.exe; image_ocr_ready=True; pymupdf=True; notes=OCR/PDF dependencies look r

## TODO: True CSV/JSON Imports

The current pipeline is for screenshots and PDFs. Add source-specific importers later for native exports, returning the same long-form observation schema used by `device_observations_long.csv`.

In [27]:
# TODO: Add Oura CSV/JSON import for Ring 4 finger and Ring 3 toe exports.
def load_oura_export(path):
    raise NotImplementedError("Parse Oura native exports into normalized observations.")


# TODO: Add Samsung Health / SleepWatch import.
def load_samsung_sleepwatch_export(path):
    raise NotImplementedError("Parse Samsung/SleepWatch native exports into normalized observations.")


# TODO: Add Muse native export or PDF table import.
def load_muse_export(path):
    raise NotImplementedError("Parse Muse exports into normalized observations.")


# TODO: Add OSCAR and SleepScope structured export import.
def load_cpap_export(path):
    raise NotImplementedError("Parse OSCAR/SleepScope exports into normalized observations.")